<!-- VIDEO: 1 / 프롬프트 엔지니어링 — Zero/One/Few-shot의 직관 -->

# 03. Prompt Engineering + Structured Output

> **학습 목표**
> 1. **Zero-shot, One-shot, Few-shot** 프롬프팅의 차이와 적용 기준을 이해한다.
> 2. `FewShotChatMessagePromptTemplate`을 사용해 예시를 체계적으로 관리한다.
> 3. `SemanticSimilarityExampleSelector`와 무료 임베딩을 결합해 입력에 적합한 예시를 자동 선택한다.
> 4. **Pydantic 기반 구조화 출력**(`with_structured_output`)으로 LLM 응답을 타입 안전한 객체로 받는다.
> 5. 한국어 후기 및 부동산 매물 데이터를 JSON으로 자동 추출하는 패턴을 익힌다.
>
> **선수 지식**: 02번 노트북(LCEL 체인 구성).

---

본 챕터는 실무 LLM 코드의 두 가지 핵심 축을 다룹니다. 하나는 모델에게 입력을 어떻게 전달할 것인가에 대한 **프롬프트 엔지니어링**이고, 다른 하나는 모델 응답을 어떻게 일관된 구조로 받을 것인가에 대한 **구조화 출력**입니다.

---

## 1. 환경 설정

이전 노트북과 동일한 패턴의 LLM 초기화 셀입니다. 보유한 키에 따라 자동으로 제공자가 분기됩니다.

In [1]:
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ✅ Option 1: Google Gemini 2.5 Flash-Lite (무료) — thinking_budget=0 으로 추론 토큰 비활성화
# llm = init_chat_model("google_genai:gemini-2.5-flash-lite", temperature=0,
#                        model_kwargs={"thinking_budget": 0})

# ✅ Option 2: Groq Qwen3 32B (무료) — reasoning_effort="none" 으로 <think> 토큰 비활성화
llm = init_chat_model("groq:qwen/qwen3-32b", temperature=0, reasoning_effort="none")

# ✅ Option 3: Ollama Gemma 4 E4B (로컬) — reasoning=False 으로 <think> 토큰 비활성화
# llm = init_chat_model("ollama:gemma4:e4b", temperature=0, reasoning=False)

# ⚙️ Option 4: OpenAI (유료) — reasoning 토큰 없음, 추가 파라미터 불필요
# llm = init_chat_model("openai:gpt-4.1-mini", temperature=0)

import re
from langchain_core.runnables import RunnableLambda

def strip_think(text: str) -> str:
    """<think>...</think> 추론 토큰 블록을 제거합니다. (안전 장치로 설정)"""
    return re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

# StrOutputParser → AIMessage.content 추출 후 <think> 블록 제거
parser = StrOutputParser() | RunnableLambda(strip_think)
print("✅ ready")

✅ ready


| 모델 | 파라미터 | 비고 |
|---|---|---|
| **Groq Qwen3** | `reasoning_effort="none"` | 허용값: `"none"` / `"default"` |
| **Gemini 2.5** | `model_kwargs={"thinking_budget": 0}` | `-1` = 동적 자동, `0` = 비활성화 |
| **Ollama** | `reasoning=False` | `None` = 모델 기본값 따름 |
| **OpenAI gpt-4.1-mini** | 추가 파라미터 불필요 | reasoning 토큰 자체가 없음 |

`strip_think` 커스텀 파서는 모델이 파라미터를 무시하거나 다른 모델로 교체 시 안전망으로 그대로 유지됩니다.

## 2. Zero-shot Prompting

예시 없이 지시만 제공하고, 모델의 사전 학습 지식만으로 작업을 수행하도록 요청하는 방식입니다. 일반 상식 질의나 단순 분류와 같이 모델이 이미 충분히 학습한 작업에 적합합니다.

In [2]:
zero_shot = ChatPromptTemplate.from_messages([
    ("system", "당신은 ABC 사내 업무 분류기입니다. 입력 문장을 "
               "[휴가/출장/IT/급여/기타] 중 하나로만 분류해 라벨만 출력하세요. /no_think"),
    ("human", "{text}"),
])

chain = zero_shot | llm | parser

samples = [
    "다음 주 수요일에 연차 쓰고 싶어요.",
    "노트북이 안 켜져요.",
    "이번 달 급여명세서는 어디서 확인하나요?",
    "부산 출장비 정산 방법이 궁금합니다.",
]
for s in samples:
    print(f"{chain.invoke({'text': s})} ← {s}")

휴가 ← 다음 주 수요일에 연차 쓰고 싶어요.
IT ← 노트북이 안 켜져요.
급여 ← 이번 달 급여명세서는 어디서 확인하나요?
출장 ← 부산 출장비 정산 방법이 궁금합니다.


## 3. One-shot Prompting

지시와 함께 **예시 하나**를 제공합니다. 출력 형식이 까다롭거나, 모델이 따라야 할 패턴이 명확히 존재할 때 결과 일관성을 높일 수 있습니다.

In [3]:
one_shot = ChatPromptTemplate.from_messages([
    ("system", "문장에서 의도와 감정을 추출해 다음 형식으로 출력하세요.\n"
               "형식: intent=<의도> | sentiment=<긍정/중립/부정>"),
    ("human", "배송이 너무 느려요, 환불해주세요."),
    ("ai",    "intent=환불요청 | sentiment=부정"),
    ("human", "{text}"),
])

chain = one_shot | llm | parser
print(chain.invoke({"text": "상품이 정말 마음에 들어요. 재구매하고 싶어요."}))
print(chain.invoke({"text": "언제쯤 도착하나요?"}))

intent=재구매의향 | sentiment=긍정
intent=배송문의 | sentiment=중립


## 4. Few-shot Prompting

복수의 예시를 제공해 모델이 패턴을 학습하도록 유도하는 방식입니다. 예시 개수에 따라 다음과 같이 구분됩니다.

| 방식 | 특징 | 적합한 작업 |
|---|---|---|
| **Zero-shot** | 예시 없이 지시만 제공 | 일반 상식, 단순 분류 |
| **One-shot** | 예시 1개 제공 | 출력 형식이 까다로운 작업 |
| **Few-shot** | 예시 여러 개 제공 | 도메인 특화 패턴, 일관성 요구 작업 |

LangChain의 `FewShotChatMessagePromptTemplate`은 system, human, ai 메시지 형식의 예시를 별도로 분리해 관리할 수 있도록 합니다. 예시 데이터와 프롬프트 형식을 분리해 유지보수성이 높아집니다.

In [4]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

examples = [
    {"input": "다음 주 금요일에 연차 쓸게요.", "output": "카테고리=휴가 | 긴급도=낮음"},
    {"input": "지금 VPN이 안 됩니다. 곧 회의인데 급해요.", "output": "카테고리=IT | 긴급도=높음"},
    {"input": "부산 KTX 티켓비 영수증 제출하려면?", "output": "카테고리=출장 | 긴급도=낮음"},
    {"input": "급여가 잘못 들어온 것 같아요.", "output": "카테고리=급여 | 긴급도=중간"},
]

example_prompt = ChatPromptTemplate.from_messages([
    ("human",  "{input}"),
    ("ai",     "{output}"),
])

few_shot = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

final = ChatPromptTemplate.from_messages([
    ("system", "사내 문의를 카테고리와 긴급도로 분류합니다. 예시를 참고해 동일 형식으로만 응답하세요."),
    few_shot,
    ("human", "{input}"),
])

chain = final | llm | parser
for q in [
    "점심 직전에 노트북이 먹통입니다. 오후 고객 미팅이 있어요.",
    "다음 달 월차 신청은 어떻게 하나요?",
    "해외 출장 환전은 어디서?",
]:
    print(f"{chain.invoke({'input': q})} ← {q}")

카테고리=IT | 긴급도=높음 ← 점심 직전에 노트북이 먹통입니다. 오후 고객 미팅이 있어요.
카테고리=휴가 | 긴급도=낮음 ← 다음 달 월차 신청은 어떻게 하나요?
카테고리=출장 | 긴급도=중간 ← 해외 출장 환전은 어디서?


<!-- VIDEO: 2 / 의미 기반 예시 선택 — Semantic Similarity -->

## 5. Example Selector — 입력에 따른 예시 동적 선택

예시 수가 많아지면 모든 예시를 매번 프롬프트에 포함시키는 방식은 토큰을 비효율적으로 소비합니다. `SemanticSimilarityExampleSelector`는 입력과 의미적으로 가까운 예시만 선별하여 포함시키므로, 비용과 지연 시간을 줄이면서 정확도는 유지할 수 있습니다.

> **임베딩 모델**: 본 강의에서는 Ollama로 로컬 실행하는 **`bge-m3`** (BAAI 다국어 임베딩, 1024차원)를 사용합니다. API 키 없이 동작하며 한국어 품질이 우수합니다.
>
> **사전 준비**: 노트북을 실행하기 전 터미널에서 `ollama pull bge-m3` 로 모델을 받아 두세요(약 1.2GB). Ollama 설치와 사용법은 [SETUP.md](./SETUP.md) 3번 섹션을 참고하세요.

In [5]:
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_ollama import OllamaEmbeddings

# 예시 풀 (입력-출력 쌍)
many_examples = [
    {"input": "연차 신청 기한", "output": "HR 시스템에서 3 영업일 전 신청"},
    {"input": "경조사 휴가 일수", "output": "본인 결혼 5일, 자녀 결혼 1일, 부모 사망 5일"},
    {"input": "국내 출장비 한도", "output": "숙박 100,000원/박, 식대 30,000원/일"},
    {"input": "해외 출장비 한도", "output": "A지역 150 USD, B지역 120 USD, C지역 100 USD"},
    {"input": "VPN 접속 방법", "output": "FortiClient + SSO + OTP"},
    {"input": "노트북 AS 신청", "output": "IT 헬프데스크 ithelp@kea.example.com 또는 내선 1700"},
    {"input": "급여 지급일", "output": "매월 25일"},
    {"input": "재택근무 신청", "output": "팀별 주 2일 가능, 팀장 사전 승인"},
]

# 로컬 Ollama 임베딩 (BAAI bge-m3, 1024차원). 사전 준비: `ollama pull bge-m3`
embeddings = OllamaEmbeddings(model="bge-m3")

selector = SemanticSimilarityExampleSelector.from_examples(
    many_examples,
    embeddings,
    vectorstore_cls=InMemoryVectorStore,
    k=2,   # 상위 2개만 선택
)

# 입력과 가장 가까운 예시 확인
query = "이번 달 월급이 어디 들어오는지 알려줘"
selected = selector.select_examples({"input": query})
print(f"질문: {query}\n")
for i, ex in enumerate(selected, 1):
    print(f"[{i}] Q: {ex['input']}\n    A: {ex['output']}")

질문: 이번 달 월급이 어디 들어오는지 알려줘

[1] Q: 급여 지급일
    A: 매월 25일
[2] Q: 재택근무 신청
    A: 팀별 주 2일 가능, 팀장 사전 승인


<!-- VIDEO: 3 / Pydantic 구조화 출력 — JSON 파싱 제거 -->

## 6. Structured Output — 스키마 기반 응답

자유 텍스트 응답은 사람이 읽기에는 적합하지만, 후속 코드에서 처리할 때는 정규식이나 JSON 파싱이 필요하며 모델이 형식을 어길 경우 오류가 발생합니다.

**Pydantic 스키마**를 사용하면 모델에게 응답할 데이터의 필드, 타입, 허용 값을 사전에 명시적으로 정의할 수 있습니다.

```mermaid
classDiagram
    class TicketClassification {
        +Literal category
        +Literal urgency
        +str summary
        +bool requires_action
    }
    note for TicketClassification "category : 휴가 | IT | 급여 | 출장<br/>urgency  : 낮음 | 중간 | 높음<br/>summary  : 20자 이내 요약<br/>requires_action : 후속 조치 필요 여부"
```

`with_structured_output(Schema)` 한 줄을 적용하면 모델 응답이 곧바로 **Pydantic 객체**로 반환되며, 별도의 파싱 단계가 필요하지 않습니다.

In [6]:
from pydantic import BaseModel, Field
from typing import Literal

class TicketClassification(BaseModel):
    """사내 업무 티켓 분류 결과"""
    category: Literal["휴가", "출장", "IT", "급여", "기타"] = Field(description="문의 카테고리")
    urgency: Literal["낮음", "중간", "높음"] = Field(description="처리 긴급도")
    summary: str = Field(description="20자 이내 요약")
    requires_manager_approval: bool = Field(description="팀장 승인 필요 여부")

structured_llm = llm.with_structured_output(TicketClassification)

ticket = structured_llm.invoke(
    "지금 VPN이 안 돼서 긴급한 고객 미팅에 접속을 못 하고 있어요. 바로 도와주세요!"
)

print(type(ticket).__name__)
print(ticket)
print()
print("카테고리:", ticket.category)
print("긴급도:", ticket.urgency)
print("요약:", ticket.summary)
print("팀장 승인 필요:", ticket.requires_manager_approval)

TicketClassification
category='IT' urgency='높음' summary='VPN 접속 문제 긴급' requires_manager_approval=False

카테고리: IT
긴급도: 높음
요약: VPN 접속 문제 긴급
팀장 승인 필요: False


### 구조화 출력의 이점

- **별도 파싱 단계 불필요**: JSON 파싱 오류 가능성을 제거합니다.
- **타입 안전성**: Pydantic 검증을 통해 런타임 오류를 사전에 감지할 수 있습니다.
- **IDE 자동완성 지원**: `.category`, `.urgency` 등 속성 접근이 가능해 개발 생산성이 향상됩니다.
- **후속 파이프라인 연계**: 반환된 객체를 그대로 DB 저장, API 호출 등 다음 단계에 전달할 수 있습니다.

---

## 7. 한국어 비즈니스 시나리오 실습

### 7.1 쇼핑몰 후기 분석 — 평점, 카테고리, 키워드 자동 추출

대량의 사용자 후기를 사람이 일일이 검토하기 어려운 상황에서, 구조화 출력 패턴을 통해 핵심 정보를 자동으로 추출합니다. 운영팀의 의사결정 속도를 향상시키는 표준 패턴입니다.

In [7]:
# (pydantic, Literal 은 위 셀에서 이미 import 됨 — 재사용)

class ReviewAnalysis(BaseModel):
    """쇼핑몰 후기 분석 결과"""
    score: int = Field(ge=1, le=5, description="추정 별점 1~5")
    category: Literal["배송", "품질", "디자인", "사이즈", "가격", "고객서비스", "기타"]
    keywords: list[str] = Field(description="핵심 키워드 (최대 3개, 3개를 넘지 마세요)")
    reply_tone: Literal["사과", "감사", "안내"] = Field(description="권장 답변 톤")
    needs_human_followup: bool = Field(description="고객 응대팀이 직접 처리 필요 여부")

review_analyzer = llm.with_structured_output(ReviewAnalysis)

reviews = [
    "배송이 너무 느렸어요. 일주일이나 걸렸네요. 제품은 괜찮은데 다음엔 다른 곳에서 살 듯",
    "디자인 진짜 예뻐요!! 친구한테 선물했는데 너무 좋아하네요 ㅎㅎ",
    "사이즈가 너무 작아요. 표기는 L인데 실제로는 M 같은 느낌. 환불 요청드립니다",
]

for r in reviews:
    result = review_analyzer.invoke(r)
    print(f"⭐ {result.score} | {result.category} | {result.reply_tone} | 직접처리: {result.needs_human_followup}")
    print(f"   키워드: {result.keywords}")
    print(f"   원문: {r[:35]}...\n")

⭐ 3 | 배송 | 사과 | 직접처리: True
   키워드: ['배송 느림', '일주일 소요', '다른 곳 고려']
   원문: 배송이 너무 느렸어요. 일주일이나 걸렸네요. 제품은 괜찮은데 다...

⭐ 5 | 디자인 | 감사 | 직접처리: False
   키워드: ['예뻐요', '선물', '좋아하네요']
   원문: 디자인 진짜 예뻐요!! 친구한테 선물했는데 너무 좋아하네요 ㅎㅎ...

⭐ 2 | 사이즈 | 사과 | 직접처리: True
   키워드: ['사이즈 작음', '환불 요청', '표기 오류']
   원문: 사이즈가 너무 작아요. 표기는 L인데 실제로는 M 같은 느낌. ...



### 7.2 부동산 매물 설명 — 구조화 데이터로 변환

자유 형식으로 작성된 매물 설명을 데이터베이스에 바로 적재할 수 있는 정형 객체로 변환합니다. 크롤링 결과나 외부 API 응답을 정규화할 때 동일한 패턴을 적용할 수 있습니다.

In [8]:
from typing import Optional

class Listing(BaseModel):
    """부동산 매물 정보"""
    region: str = Field(description="시/구/동")
    deal_type: Literal["매매", "전세", "월세"]
    price_won: int = Field(description="가격을 '원 단위 정수'로만 표기 (예: 350000000). 한국어 표현(3억5천)은 모두 정수로 변환. 월세는 보증금 기준")
    monthly_rent_won: Optional[int] = Field(default=None, description="월세인 경우 월 임대료 (원 단위 정수)")
    area_pyeong: float = Field(description="평수(소수 1자리)")
    rooms: int = Field(description="방 개수")
    floor: Optional[str] = Field(default=None, description="층수 정보(예: '15층/25층')")
    move_in: str = Field(description="입주 가능 시점")
    highlights: list[str] = Field(description="주요 특징 (최대 3개)")

listing_parser = llm.with_structured_output(Listing)

raw = """
강남구 역삼동 신축 오피스텔 전세 매물입니다. 보증금 3억 5천만원, 18평형 방2개 화장실1개 구조,
15층 매물로 한강 일부 조망 가능합니다. 즉시입주 가능하며 풀옵션(냉장고/세탁기/에어컨/인덕션)
구비. 2호선 역삼역 도보 5분, 신축 5년차로 깨끗합니다.
"""

result = listing_parser.invoke(raw)
print(result.model_dump_json(indent=2))

{
  "region": "강남구 역삼동",
  "deal_type": "전세",
  "price_won": 350000000,
  "monthly_rent_won": null,
  "area_pyeong": 18.0,
  "rooms": 2,
  "floor": "15층",
  "move_in": "즉시입주",
  "highlights": [
    "신축",
    "한강 조망",
    "풀옵션"
  ]
}


---

## 실습 과제: 감정 및 긴급도 분석기

다음 요구사항을 만족하는 구조화 분석기를 직접 구현해 보세요.

**요구사항**

1. Pydantic `BaseModel`로 `sentiment`(긍정/중립/부정), `urgency`(1~5 정수), `action_suggestion`(10자 이내 행동 제안) 필드를 가진 스키마를 정의합니다.
2. `with_structured_output`을 사용해 모델에 스키마를 연결합니다.
3. 예시 문장 3개 이상을 입력하여 분류 결과를 확인합니다.

In [9]:
# 여기에 코드를 작성하세요
# Hint: from pydantic import BaseModel, Field
#       class Analysis(BaseModel): sentiment: str; urgency: int; action_suggestion: str


<details>
<summary>모범 답안 보기</summary>

```python
from pydantic import BaseModel, Field
from typing import Literal

class Analysis(BaseModel):
    sentiment: Literal["긍정", "중립", "부정"]
    urgency: int = Field(ge=1, le=5, description="1(낮음)~5(긴급)")
    action_suggestion: str = Field(max_length=20, description="추천 행동")

analyzer = llm.with_structured_output(Analysis)

for msg in [
    "VPN 안 돼요! 10분 뒤 고객 회의라구요",
    "이번 달 경비 청구 마감은 언젠가요?",
    "정말 친절하게 응대해주셔서 감사합니다.",
]:
    r = analyzer.invoke(msg)
    print(f"[{r.sentiment} / 긴급도 {r.urgency}] {r.action_suggestion}  ← {msg}")
```

</details>

---

## 트러블슈팅

| 증상 | 원인 | 해결 방법 |
|---|---|---|
| `ValidationError`: 필드 누락 | 모델이 일부 필드를 생성하지 않음 | `Field(description="...")`에 필드 의미를 상세히 기술 |
| 동일 입력에 결과가 매번 달라짐 | `temperature` 값이 0이 아님 | 분류·추출 작업은 `temperature=0`으로 고정 |
| `with_structured_output` 미지원 | 일부 로컬 모델 또는 구버전 통합 | Gemini 또는 Groq Qwen3 사용 권장 |
| Few-shot 예시가 반영되지 않음 | 예시 형식 불일치 | `example_prompt`의 입력/출력 키와 `examples` 딕셔너리 키가 정확히 일치하는지 확인 |
| 임베딩 호출 실패 (`Connection refused`) | Ollama 서비스 미실행 | Ollama 앱 실행 후 `ollama list` 로 `bge-m3` 보유 확인. 미보유 시 `ollama pull bge-m3` |
| 임베딩 차원 불일치 (재로드 시) | 인덱스 생성 시와 다른 임베딩 모델 사용 | 동일 모델(`bge-m3`, 1024차원)을 유지하거나 인덱스를 재생성 |

---

## 마무리

본 노트북에서 학습한 내용은 다음과 같습니다.

- Zero-shot, One-shot, Few-shot 프롬프팅의 차이와 적용 기준
- `FewShotChatMessagePromptTemplate`을 활용한 예시 관리
- `SemanticSimilarityExampleSelector`를 활용한 토큰 절감과 정확도 향상
- Pydantic 스키마와 `with_structured_output`을 활용한 타입 안전한 응답 처리
- 한국어 후기 및 부동산 매물 자동 구조화 사례

### 다음 노트북(04번) 예고

지금까지의 LLM 호출은 모델이 사전 학습한 지식만을 활용했습니다. 그러나 "현재 서울 날씨는?" 또는 "이 회사의 주가는?"과 같이 **실시간 정보**나 **외부 시스템 조회**가 필요한 질의에는 응답할 수 없습니다.

다음 노트북에서는 **Tools**를 다룹니다. 웹 검색, 계산기, 데이터베이스 조회 등 외부 시스템과 LLM을 연결하는 표준 인터페이스입니다.